# Forecast
## Data Preparation 

* Use the [Epicurious dataset](https://drive.google.com/file/d/1hzmxNBrY7-9mv5EpqAvhVUiJahfrcYUN/view?usp=sharing) collected by HugoDarwood.
* Filter the columns; the fewer non-ingredient columns in your dataset, the better. You will predict the rating or rating category using only the ingredients. 
  


In [1]:
import pandas as pd
import numpy as np

from typing import List, Dict, Tuple, Optional

import math

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_squared_error, accuracy_score,
    precision_score, make_scorer
)
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    RandomForestClassifier, ExtraTreesClassifier
)

from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm 

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

#from rapidfuzz import process, utils
import re


import joblib

RANDOM_STATE = 21

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 50)

### Загрузка исходного датасета `epi_r.csv`


In [ ]:
DATA_PATH = "data/epi_r.csv"

df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
df_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: '../datasets/epi_r.csv'

In [ ]:
columns_df = pd.DataFrame({
    "column_name": df_raw.columns
})

columns_df.head()

,column_name
0,title
1,rating
2,calories
3,protein
4,fat



Запись в **epi_columns.csv**:

In [ ]:
COLUMNS_PATH = "epi_columns.csv"
columns_df.to_csv(COLUMNS_PATH, index=False)

COLUMNS_PATH


'epi_columns.csv'

### -> Отдаем ИИ (Perplexity) чтобы подготовить список **мусорных колонок**
Вот результат его работы:

In [ ]:
META_DROP = {"calories", "protein", "fat", "sodium"}

GEOGRAPHY_TAGS = [
	"alabama", "alaska", "arizona", "aspen", "atlanta",
	"australia", "beverly hills", "boston", "brooklyn",
	"cambridge", "california", "chicago", "colorado",
	"columbus", "connecticut", "costa mesa", "dallas",
	"denver", "florida", "georgia", "hawaii", "healdsburg",
	"hollywood", "houston", "idaho", "illinois", "indiana",
	"iowa", "kansas", "kansas city", "kentucky", "lancaster",
	"las vegas", "louisiana", "louisville", "maine",
	"maryland", "massachusetts", "michigan", "minneapolis",
	"minnesota", "mississippi", "missouri", "nebraska",
	"new hampshire", "new jersey", "new mexico", "new orleans",
	"new york", "ohio", "oklahoma", "oregon", "pacific palisades",
	"pennsylvania", "pittsburgh", "portland", "providence",
	"rhode island", "san francisco", "santa monica", "seattle",
	"south carolina", "st. louis", "tennessee", "texas",
	"utah", "vermont", "virginia", "washington",
	"washington, d.c.", "west virginia", "westwood", "wisconsin"
]

EVENT_TAGS = [
    "#cakeweek", "#wasteless", "anniversary", "back to school", "backyard bbq", 
    "birthday", "buffet", "christmas", "christmas eve", "cookbook critic", 
    "diwali", "easter", "engagement party", "family reunion", "father's day", 
    "flaming hot summer", "fourth of july", "friendsgiving", "graduation", 
    "halloween", "hanukkah", "kentucky derby", "kitchen olympics", "kwanzaa", 
    "labor day", "lunar new year", "mardi gras", "mother's day", "new year's day", 
    "new year's eve", "oktoberfest", "oscars", "passover", "persian new year", 
    "poker/game night", "potluck", "purim", "ramadan", "rosh hashanah/yom kippur", 
    "shavuot", "snack week", "st. patrick's day", "super bowl", "sukkot", 
    "tailgating", "thanksgiving", "valentine's day", "wedding"
]


DIET_TAGS = [
"dairy", "dairy free", "fat free", "gluten-free", "high fiber",
"kid-friendly", "kidney friendly", "kosher", "kosher for passover",
"low cal", "low carb", "low cholesterol", "low fat", "low sodium",
"low sugar", "low/no sugar", "no meat, no problem", "no sugar added",
"paleo", "peanut free", "soy free", "sugar conscious", "tree nut free",
"vegan", "vegetarian", "wheat/gluten-free"
]

COOK_TAGS = [
"22-minute meals", "3-ingredient recipes", "advance prep required", "bake",
"blender", "boil", "braise", "broil", "deep-fry", "double boiler",
"food processor", "freeze/chill", "fry", "grill", "grill/barbecue",
"mandoline", "marinate", "microwave", "mixer", "mortar and pestle",
"no-cook", "one-pot meal", "pan-fry", "pasta maker", "pressure cooker",
"quick & easy", "quick and healthy", "raw", "roast", "simmer",
"skewer", "slow cooker", "smoker", "steam", "stir-fry", "wok"
]

MISC_TAGS = [
"30 days of groceries", "alcoholic", "aperitif", "buffet", "cookbook critic",
"cook like a diner", "digestif", "edible gift", "epi + ushg",
"epi loves the microwave", "flaming hot summer", "food processor",
"frankenrecipe", "freezer food", "gourmet", "house & garden",
"house cocktail", "leftovers", "parade", "poker/game night",
"potluck", "quick and healthy", "snack", "snack week",
"tested & improved", "weelicious"
]

NON_INGREDIENT_TAGS = [
    "appetizer", "breakfast", "brunch", "dessert", "dinner", "dip", 
    "entertaining", "hors d'oeuvre", "lunch", "party", "side",
    
    "bake", "boil", "braise", "broil", "candy thermometer", "coffee grinder",
    "double boiler", "freeze/chill", "juicer", "mandoline", "marinate",
    
    "anthony bourdain", "bastille day", "bulgaria", "camping", "canada",
    "cinco de mayo", "dominican republic", "dorie greenspan", "egypt",
    "emeril lagasse", "england", "france", "germany", "guam", "haiti",
    "ireland", "israel", "italy", "jamaica", "japan", "london", 
    "los angeles", "mexico", "miami", "nancy silverton", "paris",
    "pasadena", "peru", "philippines", "spain", "suzanne goin", "switzerland",
    
    "healthy", "pescatarian",

    "condiment", "condiment/spread", "entertaining", "harpercollins", 
    "self", "shower"
]

REST = [
	"bon appétit", "bon app��tit", "cocktail", "cocktail party", "sandwich theory", "windsor", "cookbooks", "phyllo/puff pastry dough", "fruit", "stew"
]

DISH_TAGS = [
"brownie", "burrito", "casserole/gratin", "cobbler/crumble", "cookie",
"cookies", "cupcake", "fritter", "frittata", "hamburger", "lasagna",
"macaroni and cheese", "meatball", "meatloaf", "muffin", "omelet",
"pancake", "pie", "pizza", "pot pie", "quiche", "sandwich", "soufflé/meringue",
"sorbet", "stuffing/dressing", "taco", "soup/stew"
]


In [ ]:
ALL_TAGS_TO_DROP = GEOGRAPHY_TAGS + EVENT_TAGS + DIET_TAGS + COOK_TAGS + MISC_TAGS + REST + DISH_TAGS + NON_INGREDIENT_TAGS
existing_columns_to_drop = [col for col in ALL_TAGS_TO_DROP if col in df_raw.columns]
df_filtered = df_raw.drop(columns=existing_columns_to_drop)
# df.drop(columns=ALL_TAGS_TO_DROP, inplace=True, errors='ignore')

print(f"Удалено колонок: {len(ALL_TAGS_TO_DROP)}")
print(f"Осталось колонок: {len(df_filtered.columns)}")

Удалено колонок: 308
Осталось колонок: 390


**Промежуточная проверка названий колонок:**

In [ ]:
columns_df_filtered = pd.DataFrame({
    "column_name": df_filtered.columns
})

columns_df_filtered.head()

COLUMNS_FILTERED_PATH = "epi_columns_filtered.csv"
columns_df_filtered.to_csv(COLUMNS_FILTERED_PATH, index=False)

COLUMNS_FILTERED_PATH


'epi_columns_filtered.csv'

**Проблемная колонка с битой кодировкой cr��me de cacao**

In [ ]:

col_bad = 'cr��me de cacao'  # с проблемной кодировкой
col_good = 'créme de cacao'  # корректная

# проверяем существование
print(f"\nСуществует ли '{col_bad}': {col_bad in df_filtered.columns}")
print(f"Существует ли '{col_good}': {col_good in df_filtered.columns}")

if col_bad in df_filtered.columns and col_good in df_filtered.columns:
    
    
    
    are_identical = (df_filtered[col_bad] == df_filtered[col_good]).all() # если идентичны
    
    # расхождения (если есть)
    if not are_identical: 
        differences = (df_filtered[col_bad] != df_filtered[col_good]).sum()
        
        
        # примеры расхождений
        diff_mask = df_filtered[col_bad] != df_filtered[col_good]
        print("\nПримеры расхождений:")
        print(df_filtered[diff_mask][[col_bad, col_good]].head())
    
    # объединяем: если хотя бы в одной колонке 1, то в целевой 1
    df_filtered[col_good] = df_filtered[[col_bad, col_good]].max(axis=1)
    
    df_filtered = df_filtered.drop(columns=[col_bad])
    
    
elif col_bad in df_filtered.columns:
    print(f"\nТолько проблемная колонка '{col_bad}' существует")
    # переименовываем
    new_name = col_bad.replace('��', 'é')
    df_filtered = df_filtered.rename(columns={col_bad: new_name})


df_filtered.to_csv('epi_recipes_filtered.csv', index=False)




Существует ли 'cr��me de cacao': True
Существует ли 'créme de cacao': True

Примеры расхождений:
      cr��me de cacao  créme de cacao
1608              0.0             1.0
3010              0.0             1.0
4672              0.0             1.0
5376              0.0             1.0
6004              0.0             1.0


### jam or jelly

In [ ]:
def split_and_merge_jam_columns(df):
    """
    Разделяет 'jam or jelly' на 'jam' и 'jelly'
    и объединяет с существующей колонкой 'jam'
    """
    df = df.copy()
    
    # 1. Проверяем, есть ли у нас составная колонка
    composite_col = None
    for col in df.columns:
        if str(col).lower() == 'jam or jelly':
            composite_col = col
            break
    
    # 2. Если есть составная колонка
    if composite_col:
        print(f"Найдена составная колонка: '{composite_col}'")
        
        # Создаем отдельные колонки
        df['jam_from_composite'] = df[composite_col]  # временная
        df['jelly'] = df[composite_col]               # новая колонка для желе
        
        # 3. Объединяем с существующей 'jam'
        if 'jam' in df.columns:
            print(f"Объединяем с существующей колонкой 'jam'")
            # Логическое ИЛИ: если есть в одной из колонок
            df['jam'] = df['jam'] | df['jam_from_composite']
        else:
            print(f"Создаем новую колонку 'jam'")
            df['jam'] = df['jam_from_composite']
        
        # 4. Удаляем временные колонки
        df = df.drop(columns=[composite_col, 'jam_from_composite'])
        print(f"Удалена составная колонка '{composite_col}'")
    
    else:
        print("Составной колонки 'jam or jelly' не найдено")
        
        # Если нет колонки 'jam', создаем пустую
        if 'jam' not in df.columns:
            df['jam'] = 0
            print("Создана пустая колонка 'jam'")
    
    return df

In [ ]:
df_filtered = split_and_merge_jam_columns(df_filtered)

Найдена составная колонка: 'jam or jelly'
Создаем новую колонку 'jam'
Удалена составная колонка 'jam or jelly'


In [ ]:
rows_with_nan = df_filtered.isna().any(axis=1).sum()
total_rows = len(df_filtered)
print(f"Строк с NaN: {rows_with_nan} из {total_rows} ({rows_with_nan/total_rows:.1%})")

Строк с NaN: 4188 из 20052 (20.9%)


In [ ]:
df_filtered = df_filtered.drop(columns=META_DROP)

In [ ]:
def expand_composite_ingredients(df):
    """
    'sweet potato/yam' -> колонки 'sweet potato' и 'yam' и тд
    """
    df_expanded = df.copy()
    columns_to_drop = []
    columns_to_add = {}
    
    for col in df.columns:
        if '/' in str(col):
            parts = str(col).split('/')
            parts = [p.strip() for p in parts]
            
            # Для каждой части создаем колонку
            for part in parts:
                if part not in df.columns:
                    columns_to_add[part] = df[col]
            
            # Помечаем исходную колонку на удаление
            columns_to_drop.append(col)
    
    # Удаляем составные колонки
    df_expanded = df_expanded.drop(columns=columns_to_drop)
    
    # Добавляем новые колонки
    for new_col, values in columns_to_add.items():
        df_expanded[new_col] = values
    
    return df_expanded

In [ ]:

# вызываем функцию
df_processed = expand_composite_ingredients(df_filtered)

df_processed

,title,rating,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,arugula,asian pear,asparagus,avocado,bacon,banana,barley,basil,bass,bean,beef,beef rib,beef shank,beef tenderloin,beer,beet,bell pepper,berry,biscuit,bitters,blackberry,blue cheese,blueberry,bok choy,bourbon,bran,brandy,bread,breadcrumbs,brie,brine,brisket,broccoli,broccoli rabe,brown rice,brussel sprout,buffalo,bulgur,butter,buttermilk,butternut squash,cabbage,cake,calvados,campari,candy,cantaloupe,capers,caraway,cardamom,carrot,cashew,cauliflower,caviar,celery,chambord,champagne,chard,chartreuse,cheddar,cheese,cherry,chestnut,chicken,chickpea,chile,chile pepper,chili,chill,chive,chocolate,cilantro,cinnamon,citrus,clam,clove,coconut,cod,coffee,collard greens,coriander,corn,cornmeal,cottage cheese,couscous,crab,cranberry,cranberry sauce,cream cheese,créme de cacao,...,sage,sake,salad,salad dressing,salmon,salsa,sangria,sardine,sauce,sausage,sauté,scallop,scotch,seafood,seed,semolina,sesame,sesame oil,shallot,shellfish,sherry,shrimp,smoothie,snapper,sour cream,sourdough,soy,soy sauce,sparkling wine,spice,spinach,spirit,spring,spritzer,squash,squid,steak,stock,strawberry,sugar snap pea,summer,swiss cheese,swordfish,tamarind,tangerine,tapioca,tarragon,tart,tea,tequila,thyme,tilapia,tofu,tomatillo,tomato,tortillas,tree nut,triple sec,tropical fruit,trout,tuna,turnip,vanilla,veal,vegetable,venison,vermouth,vinegar,vodka,waffle,walnut,wasabi,watercress,watermelon,whiskey,white wine,whole wheat,wild rice,wine,winter,yellow squash,yogurt,yonkers,yuca,zucchini,turkey,jelly,jam,butterscotch,caramel,cognac,armagnac,green onion,scallion,hominy,masa,milk,cream,sweet potato,yam
0,"Lentil, Apple, and Turkey Wrap",2.500,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Potato and Fennel Soup Hodge,3.750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

### Поиск строк-дубликатов:

In [ ]:

def find_duplicate_rows(df):
    """Найти полностью одинаковые строки в бинарном DataFrame"""
    # преобразуем в строку для быстрого сравнения
    df_str = df.astype(str).agg(''.join, axis=1)
    
    # находим дубликаты
    duplicates = df_str[df_str.duplicated(keep=False)]
    
    print(f"Всего строк: {len(df)}")
    print(f"Уникальных комбинаций: {df_str.nunique()}")
    print(f"Дубликатов: {len(df) - df_str.nunique()}")
    
    # показать примеры
    if len(duplicates) > 0:
        # покажем первую дублирующуюся комбинацию
        sample_dup = duplicates.iloc[0]
        dup_indices = df_str[df_str == sample_dup].index.tolist()
        print(f"\nПример дубликатов (индексы {dup_indices}):")
        print(df.loc[dup_indices].head())
    
    return duplicates

duplicates = find_duplicate_rows(df_processed)

Всего строк: 20052
Уникальных комбинаций: 18239
Дубликатов: 1813

Пример дубликатов (индексы [14, 19265]):
                title  rating  almond  amaretto  anchovy  anise  apple  \
14     Peach Mustard    3.125     0.0       0.0      0.0    0.0    0.0   
19265  Peach Mustard    3.125     0.0       0.0      0.0    0.0    0.0   

       apple juice  apricot  artichoke  arugula  asian pear  asparagus  \
14             0.0      0.0        0.0      0.0         0.0        0.0   
19265          0.0      0.0        0.0      0.0         0.0        0.0   

       avocado  bacon  banana  barley  basil  bass  bean  beef  beef rib  \
14         0.0    0.0     0.0     0.0    0.0   0.0   0.0   0.0       0.0   
19265      0.0    0.0     0.0     0.0    0.0   0.0   0.0   0.0       0.0   

       beef shank  beef tenderloin  beer  beet  bell pepper  berry  biscuit  \
14            0.0              0.0   0.0   0.0          0.0    0.0      0.0   
19265         0.0              0.0   0.0   0.0          0.0 

In [ ]:
df_processed = df_processed.drop_duplicates()

print(f"После удаления дубликатов: {df_processed.shape}")

# Проверим, что дубликатов больше нет
duplicates_check = df_processed.duplicated().sum()
print(f"Осталось дубликатов: {duplicates_check}")

После удаления дубликатов: (18239, 392)
Осталось дубликатов: 0


In [ ]:
df_processed.to_csv('epi_recipes_processed.csv', index=False)


На этом шаге **ничего не чистим**, только смотрим.

## Быстрый аудит колонок


In [ ]:

# типы колонок
df_processed.dtypes.value_counts()


float64    391
object       1
dtype: int64

In [ ]:


# процент пропусков
(df_processed.isna().mean()
 .sort_values(ascending=False)
 .head(20))

title                0.0
punch                0.0
prune                0.0
prosciutto           0.0
poultry sausage      0.0
poultry              0.0
potato salad         0.0
potato               0.0
port                 0.0
pork tenderloin      0.0
pork rib             0.0
pork chop            0.0
pork                 0.0
poppy                0.0
pomegranate juice    0.0
pomegranate          0.0
poblano              0.0
poach                0.0
plum                 0.0
plantain             0.0
dtype: float64

## Подготавливаем переменные для тренировки моделей

In [ ]:
# продолжаем работать с df_processed
df = df_processed.copy()
df = df.drop(columns=['rating', 'title']).astype(np.int8)
y_rating = df_processed['rating'].astype(float)

df.shape, y_rating.describe()


((18239, 390),
 count    18239.000000
 mean         3.713368
 std          1.334082
 min          0.000000
 25%          3.750000
 50%          4.375000
 75%          4.375000
 max          5.000000
 Name: rating, dtype: float64)

### Самопроверка по Чек-листу

In [ ]:
filtered_cols = df.columns.to_list()
need_present = ['milk', 'jam']
need_absent = ["title"] + ALL_TAGS_TO_DROP + ["protein"]  # protein — метаданные
present_ok = all(x in [str(c).lower() for c in filtered_cols] for x in need_present)
absent_ok = all(x not in [str(c).lower() for c in filtered_cols] for x in need_absent)

len(filtered_cols), present_ok, absent_ok

(390, True, True)

## Функция преобразования рецептов в текст

Финальная модель в приложении удобно работает как `Pipeline(TF-IDF -> classifier)` и принимает список строк.
Каждая строка — это ингредиенты рецепта, склеенные через пробел.


In [ ]:
def build_ingredients_text(X: pd.DataFrame) -> List[str]:
    cols = list(X.columns)
    arr = X.to_numpy(dtype=np.int8)
    out: List[str] = []
    for i in range(arr.shape[0]):
        idx = np.where(arr[i] == 1)[0]
        tokens = [cols[j] for j in idx]
        out.append(" ".join(tokens))
    return out

## Regression: RMSE

Смотрим Baseline (среднее) и пробуем несколько моделей с подбором гиперпараметров.
Проводим сравнение.


In [ ]:
# делим данные на обучающие и тестовые
train_idx, test_idx = train_test_split(
    df.index, 
    test_size=0.2, 
    random_state=RANDOM_STATE
)

# создаем тексты из ингридиентов ('apple bean lentil lettuce tomato vegetable turkey')
texts = build_ingredients_text(df)
len(texts), texts[0][:120]
def make_texts(df):
    cols = list(df.columns)
    arr = df.to_numpy(dtype=np.int8)
    texts = []
    for i in range(arr.shape[0]):
        idx = np.where(arr[i] == 1)[0]
        tokens = [cols[j] for j in idx]
        texts.append(" ".join(tokens))
    return texts


X_train_txt = make_texts(df.loc[train_idx])
X_test_txt = make_texts(df.loc[test_idx])
y_train, y_test = y_rating[train_idx], y_rating[test_idx]

print(f"Обучающих рецептов: {len(X_train_txt)}")
print(f"Тестовых рецептов: {len(X_test_txt)}")


# базовая модель (просто среднее)
print("\n" + "="*50)
print("1: Baseline Базовая модель - просто среднее")
print("="*50)
def rmse(y_true, y_pred):
    return float(math.sqrt(mean_squared_error(y_true, y_pred)))

naive_reg = DummyRegressor(strategy="mean")
dummy_train = ["dummy"] * len(X_train_txt)
dummy_test = ["dummy"] * len(X_test_txt)
naive_reg.fit(dummy_train, y_train)
rmse_naive = rmse(y_test, naive_reg.predict(dummy_test))
print(f"✓ Baseline RMSE = {rmse_naive:.4f}")

# модель RIDGE must-have по заданию
print("\n" + "="*50)
print("2: Обучаем модель Ridge")
print("="*50)

ridge_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b[\w\-]+\b")),
    ("ridge", Ridge(random_state=RANDOM_STATE)),
])

ridge_grid = {
    "tfidf__min_df": [2, 5],
    "ridge__alpha": [1.0, 10.0],
}

ridge_gs = GridSearchCV(
    ridge_pipe, 
    ridge_grid, 
    cv=3, 
    scoring="neg_root_mean_squared_error", 
    n_jobs=-1,
    verbose=1
)

ridge_gs.fit(X_train_txt, y_train)
best_ridge = ridge_gs.best_estimator_
rmse_ridge = rmse(y_test, best_ridge.predict(X_test_txt))
print(f"✓ Ridge RMSE = {rmse_ridge:.4f}")
print(f"✓ Лучшие параметры: {ridge_gs.best_params_}")

# 6. ВТОРАЯ МОДЕЛЬ (RANDOM FOREST) - МОЖНО ПРОПУСТИТЬ ЕСЛИ МЕДЛЕННО
print("\n" + "="*50)
print("3: Обучаем модель Random Forest")
print("="*50)

# Упрощаем для скорости
rf_reg_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        token_pattern=r"(?u)\b[\w\-]+\b", 
        max_features=1000
    )),
    ("rf", RandomForestRegressor(
        random_state=RANDOM_STATE, 
        n_jobs=-1,
        n_estimators=50  # МАЛО для скорости
    )),
])

rf_reg_grid = {
    "tfidf__min_df": [2],
    "rf__max_depth": [20],
}

rf_reg_gs = GridSearchCV(
    rf_reg_pipe, 
    rf_reg_grid, 
    cv=2,  # МАЛО для скорости
    scoring="neg_root_mean_squared_error", 
    n_jobs=-1,
    verbose=1
)

print("Это может занять 2-5 минут...")
rf_reg_gs.fit(X_train_txt, y_train)
best_rf_reg = rf_reg_gs.best_estimator_
rmse_rf_reg = rmse(y_test, best_rf_reg.predict(X_test_txt))
print(f"✓ Random Forest RMSE = {rmse_rf_reg:.4f}")

# 7. ИТОГИ
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ:")
print("="*50)
print(f"1. Baseline (просто среднее):  {rmse_naive:.4f}")
print(f"2. Ridge модель:                {rmse_ridge:.4f}")
print(f"3. Random Forest:               {rmse_rf_reg:.4f}")
print("="*50)


Обучающих рецептов: 14591
Тестовых рецептов: 3648

1: Baseline Базовая модель - просто среднее
✓ Baseline RMSE = 1.3450

2: Обучаем модель Ridge
Fitting 3 folds for each of 4 candidates, totalling 12 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:   11.5s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.


✓ Ridge RMSE = 1.2799
✓ Лучшие параметры: {'ridge__alpha': 10.0, 'tfidf__min_df': 2}

3: Обучаем модель Random Forest
Это может занять 2-5 минут...
Fitting 2 folds for each of 1 candidates, totalling 2 fits


[Parallel(n_jobs=-1)]: Done   2 out of   2 | elapsed:    4.0s finished


✓ Random Forest RMSE = 1.2897

РЕЗУЛЬТАТЫ:
1. Baseline (просто среднее):  1.3450
2. Ridge модель:                1.2799
3. Random Forest:               1.2897
✅ УРА! Модель работает лучше чем простое среднее!


## Classification: 6 классов (0..5) и 3 класса (bad / so-so / great)

Сначала:
- округляем рейтинг до целого 0..5;
- считаем baseline (most_frequent);
- пробуем несколько моделей с GridSearch.

Затем:
- переводим 0..5 в 3 класса:
  - bad: 0–1
  - so-so: 2–3
  - great: 4–5
- дополнительно используем метрику, где важно качество для класса **great**: precision(great).


In [ ]:
# 6-class classification (0..5)
y_round = np.rint(np.asarray(y_rating, dtype=float)).astype(int)
y_round = np.clip(y_round, 0, 5)

# иногда стратификация падает, если какой-то класс слишком редкий
try:
    X6_train, X6_test, y6_train, y6_test = train_test_split(
        texts, y_round, test_size=0.2, random_state=RANDOM_STATE, stratify=y_round
    )
except ValueError:
    X6_train, X6_test, y6_train, y6_test = train_test_split(
        texts, y_round, test_size=0.2, random_state=RANDOM_STATE
    )

# --- Naive baseline ---
naive6 = DummyClassifier(strategy="most_frequent")
naive6.fit(np.zeros((len(y6_train), 1)), y6_train)
acc6_naive = accuracy_score(y6_test, naive6.predict(np.zeros((len(y6_test), 1))))

# --- Logistic Regression (robust init for different sklearn versions) ---
try:
    lr6 = LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        multi_class="multinomial",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
except TypeError:
    # если ваша версия sklearn не принимает multi_class / n_jobs
    lr6 = LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )

logreg6 = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b[\w\-]+\b")),
    ("clf", lr6),
])

logreg6_grid = {
    "tfidf__min_df": [2, 5],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.5, 1.0, 2.0],
}

logreg6_gs = GridSearchCV(logreg6, logreg6_grid, cv=3, scoring="accuracy", n_jobs=-1)
logreg6_gs.fit(X6_train, y6_train)
best_logreg6 = logreg6_gs.best_estimator_
acc6_logreg = accuracy_score(y6_test, best_logreg6.predict(X6_test))

# --- LinearSVC ---
svc6 = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b[\w\-]+\b", ngram_range=(1, 2))),
    ("clf", LinearSVC(random_state=RANDOM_STATE)),
])
svc6_grid = {"tfidf__min_df": [2, 5], "clf__C": [0.5, 1.0, 2.0]}
svc6_gs = GridSearchCV(svc6, svc6_grid, cv=3, scoring="accuracy", n_jobs=-1)
svc6_gs.fit(X6_train, y6_train)
best_svc6 = svc6_gs.best_estimator_
acc6_svc = accuracy_score(y6_test, best_svc6.predict(X6_test))

# --- RandomForest ---
rf6 = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b[\w\-]+\b", max_features=5000)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])
rf6_grid = {"tfidf__min_df": [2, 5], "clf__n_estimators": [300, 600], "clf__max_depth": [None, 40]}
rf6_gs = GridSearchCV(rf6, rf6_grid, cv=3, scoring="accuracy", n_jobs=-1)
rf6_gs.fit(X6_train, y6_train)
best_rf6 = rf6_gs.best_estimator_
acc6_rf = accuracy_score(y6_test, best_rf6.predict(X6_test))

acc6_naive, acc6_logreg, acc6_svc, acc6_rf



(0.6641995614035088,
 0.6779057017543859,
 0.6672149122807017,
 0.6776315789473685)

---


Может выполняться минут 13:

In [ ]:
def to_3class(y_int: np.ndarray) -> np.ndarray:
    y_int = np.asarray(y_int, dtype=int)
    out = np.empty_like(y_int, dtype=object)
    out[y_int <= 1] = "bad"
    out[(y_int >= 2) & (y_int <= 3)] = "so-so"
    out[y_int >= 4] = "great"
    return out

y3 = to_3class(y_round)

X3_train, X3_test, y3_train, y3_test = train_test_split(
    texts, y3, test_size=0.2, random_state=RANDOM_STATE, stratify=y3
)

naive3 = DummyClassifier(strategy="most_frequent")
naive3.fit(np.zeros((len(y3_train), 1)), y3_train)
acc3_naive = accuracy_score(y3_test, naive3.predict(np.zeros((len(y3_test), 1))))

# метрика: precision для класса "great" (скаляр)
def _great_precision(y_true, y_pred) -> float:
    return float(precision_score(y_true, y_pred, labels=["great"], average=None, zero_division=0)[0])

great_prec = make_scorer(_great_precision)

# Модель 1: LogisticRegression
logreg3 = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b[\w\-]+\b")),
    ("clf", LogisticRegression(max_iter=3000, solver="lbfgs", n_jobs=-1)),
])
logreg3_grid = {
    "tfidf__min_df": [2, 5],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.5, 1.0, 2.0],
}
logreg3_gs = GridSearchCV(logreg3, logreg3_grid, cv=3, scoring=great_prec, n_jobs=-1)
logreg3_gs.fit(X3_train, y3_train)
best_logreg3 = logreg3_gs.best_estimator_

# Модель 2: LinearSVC
svc3 = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b[\w\-]+\b", ngram_range=(1, 2))),
    ("clf", LinearSVC(random_state=RANDOM_STATE)),
])
svc3_grid = {"tfidf__min_df": [2, 5], "clf__C": [0.5, 1.0, 2.0]}
svc3_gs = GridSearchCV(svc3, svc3_grid, cv=3, scoring=great_prec, n_jobs=-1)
svc3_gs.fit(X3_train, y3_train)
best_svc3 = svc3_gs.best_estimator_

# Модель 3 (ансамбль): ExtraTreesClassifier
etr3 = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b[\w\-]+\b", max_features=6000)),
    ("clf", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])
etr3_grid = {"tfidf__min_df": [2, 5], "clf__n_estimators": [400, 800], "clf__max_depth": [None, 40]}
etr3_gs = GridSearchCV(etr3, etr3_grid, cv=3, scoring=great_prec, n_jobs=-1)
etr3_gs.fit(X3_train, y3_train)
best_etr3 = etr3_gs.best_estimator_

# тестовые оценки (понять, что всё работает)
def great_precision_on_test(model) -> float:
    pred = model.predict(X3_test)
    # precision_score с labels=["great"] вернёт массив из 1 элемента
    return float(precision_score(y3_test, pred, labels=["great"], average=None, zero_division=0)[0])

acc3_logreg = accuracy_score(y3_test, best_logreg3.predict(X3_test))
acc3_svc = accuracy_score(y3_test, best_svc3.predict(X3_test))
acc3_etr = accuracy_score(y3_test, best_etr3.predict(X3_test))

gp_logreg = great_precision_on_test(best_logreg3)
gp_svc = great_precision_on_test(best_svc3)
gp_etr = great_precision_on_test(best_etr3)

rf3 = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b[\w\-]+\b", max_features=5000)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])
rf3_grid = {
    "tfidf__min_df": [2, 5],
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 30]
}
rf3_gs = GridSearchCV(rf3, rf3_grid, cv=3, scoring=great_prec, n_jobs=-1)
rf3_gs.fit(X3_train, y3_train)
best_rf3 = rf3_gs.best_estimator_

# --- Ансамбль 3: GradientBoostingClassifier ---
gb3 = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b[\w\-]+\b", max_features=4000)),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])
gb3_grid = {
    "tfidf__min_df": [2, 5],
    "clf__n_estimators": [100, 200], # бустинг дольше обучается, ставим меньше итераций
    "clf__learning_rate": [0.1, 0.2],
}
gb3_gs = GridSearchCV(gb3, gb3_grid, cv=3, scoring=great_prec, n_jobs=-1)
gb3_gs.fit(X3_train, y3_train)
best_gb3 = gb3_gs.best_estimator_

# Сбор всех результатов 

# Accuracy
acc3_rf = accuracy_score(y3_test, best_rf3.predict(X3_test))
acc3_gb = accuracy_score(y3_test, best_gb3.predict(X3_test))


gp_rf = great_precision_on_test(best_rf3)
gp_gb = great_precision_on_test(best_gb3)

# Итоговый вывод (кортеж из Accuracy и Precision для всех моделей)

results3 = {
    "Naive (Baseline)":      {"acc": acc3_naive,  "prec": 0.0},
    "Logistic Regression":   {"acc": acc3_logreg, "prec": gp_logreg},
    "Linear SVC":            {"acc": acc3_svc,    "prec": gp_svc},
    "ExtraTrees (Ens 1)":    {"acc": acc3_etr,    "prec": gp_etr},
    "RandomForest (Ens 2)":  {"acc": acc3_rf,     "prec": gp_rf},
    "GradientBoosting (Ens 3)": {"acc": acc3_gb,  "prec": gp_gb},
}

print(f"{'Model Name':<25} | {'Accuracy':<10} | {'Great Precision':<15}")
print("-" * 55)

for name, metrics in results3.items():
    print(f"{name:<25} | {metrics['acc']:<10.4f} | {metrics['prec']:<15.4f}")

Model Name                | Accuracy   | Great Precision
-------------------------------------------------------
Naive (Baseline)          | 0.7944     | 0.0000         
Logistic Regression       | 0.7977     | 0.8101         
Linear SVC                | 0.7826     | 0.8125         
ExtraTrees (Ens 1)        | 0.7908     | 0.8120         
RandomForest (Ens 2)      | 0.7895     | 0.8104         
GradientBoosting (Ens 3)  | 0.7919     | 0.8097         


## Выбор финальной модели и сохранение

Для CLI удобнее и устойчивее использовать **классификацию 3 класса**.
Сохраняем `best_model.joblib` в `src/data/`.

In [ ]:
# выбираем лучшую по precision(great) на тесте
cands = [
    ("logreg3", best_logreg3, gp_logreg),
    ("svc3", best_svc3, gp_svc),
    ("etr3", best_etr3, gp_etr),
    ("rf3", best_rf3, gp_rf),
    ("rf3", best_gb3, gp_gb)
]
cands_sorted = sorted(cands, key=lambda x: x[2], reverse=True)
best_name, final_model, _ = cands_sorted[0]

# обучаем на всём датасете
final_model.fit(texts, y3)

out_dir = "data/"  # обычно src/data
model_path = out_dir + "best_model.joblib"
joblib.dump(final_model, model_path)

best_name, model_path


('svc3', 'data/best_model.joblib')

## Лучшая модель - Support Vector Machine.

Она дает более точный и полезный прогноз для конечного пользователя.

Она значительно лучше «бьет» базовую (наивную) модель, чем регрессия.


## Nutrition Facts: таблица в % Daily Value

Чтобы CLI стабильно показывал %DV, сохраняем `nutrition_dv.csv` в `src/data/`.

Минимальный набор, который должен быть закрыт для проверки:
- bread
- avocado
- cheese
- garlic

Значения могут быть расширены позже, но главное — формат и наличие %DV.


In [ ]:
ingr = pd.read_csv('../FoodData_Central_foundation_food_csv_2025-12-18/food.csv')
ingr.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
2,319876,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
3,319877,sub_sample_food,Hummus,16.0,2019-04-01
4,319878,sub_sample_food,Hummus,16.0,2019-04-01


In [ ]:
nutr = pd.read_csv('../FoodData_Central_foundation_food_csv_2025-12-18/nutrient.csv')
nutr.head()

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
2,1001,Solids,G,201.0,200.0
3,1002,Nitrogen,G,202.0,500.0
4,1003,Protein,G,203.0,600.0


In [ ]:
gr = pd.read_csv('../FoodData_Central_foundation_food_csv_2025-12-18/food_nutrient.csv')
gr.head()

/Users/annabagina/Downloads/DSB12_Food_nutrition.ID_1577655-Team_TL_gilmaref.aacda10a_1488_450f-1-develop/.venv/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3166: DtypeWarning: Columns (9) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_combined = pd.merge(
    gr, 
    ingr[['fdc_id', 'data_type', 'description']], 
    on='fdc_id', 
    how='left'
)

df_combined['nutrient_id'] = pd.to_numeric(df_combined['nutrient_id'], errors='coerce')
nutr['id'] = pd.to_numeric(nutr['id'], errors='coerce')

# 2. Добавляем информацию о самих нутриентах из таблицы nutr
# В nutr ключ называется 'id', а в нашей таблице - 'nutrient_id'
df_final = pd.merge(
    df_combined, 
    nutr[['id', 'name', 'unit_name']], 
    left_on='nutrient_id', 
    right_on='id', 
    how='left'
)

# Удаляем дублирующий столбец 'id' после слияния
#df_final = df_final.drop(columns=['id'])

# Выведем результат (основные колонки для проверки)
print(df_final[['description', 'name', 'amount', 'unit_name']].head())

  description               name  amount unit_name
0      Hummus              Water   56.30         G
1      Hummus           Nitrogen    1.28         G
2      Hummus  Total lipid (fat)   19.00         G
3      Hummus                Ash    1.98         G
4      Hummus      Phosphorus, P  188.00        MG


In [ ]:
df_final = df_final[['description', 'name', 'amount']]
df_final

,description,name,amount
0,Hummus,Water,56.30
1,Hummus,Nitrogen,1.28
2,Hummus,Total lipid (fat),19.00
3,Hummus,Ash,1.98
4,Hummus,"Phosphorus, P",188.00
...,...,...,...
159280,"Shallots, bulb, peeled, root removed, raw","Fiber, total dietary",2.30
159281,"Shallots, bulb, peeled, root removed, raw","Fiber, total dietary",2.30
159282,"Shallots, bulb, peeled, root removed, raw","Fiber, total dietary",2.50
159283,"Shallots, bulb, peeled, root removed, raw","Fiber, total dietary",2.20


In [ ]:
dv_adults = {
    "Total carbohydrates": 275.0,
    "Sodium": 2.3,
    "Dietary fiber": 28.0,
    "Added sugars": 50.0,
    "Fat": 78.0,
    "Saturated fat": 20.0,
    "Cholesterol": 300.0,
    "Carbohydrate, by difference": 275.0,
    "Fiber, total dietary": 28.0,
    "Protein": 50.0,

    "Vitamin A, RAE": 900.0,             
    "Vitamin C, total ascorbic acid": 90.0, 
    "Vitamin D (D2 + D3)": 20.0,        
    "Vitamin E (alpha-tocopherol)": 15.0,
    "Vitamin K (phylloquinone)": 120,  
    "Thiamin": 1.2,                     
    "Riboflavin": 1.3,                  
    "Niacin": 1600.0,                       
    "Vitamin B-6": 1.7,                
    "Folate, DFE": 0.4,
    "Vitamin B-12": 2.4,
    "Biotin": 30.0,
    "Pantothenic acid": 5.0,
    "Choline, total": 550.0,
    
    "Calcium, Ca": 1300,                    
    "Iron, Fe": 18.0,
    "Phosphorus, P": 1250,                 
    "Iodine, I": 150.0,                  
    "Magnesium, Mg": 420.0,                 
    "Zinc, Zn": 11.0,                     
    "Selenium, Se": 55.0,              
    "Copper, Cu": 0.9,                  
    "Manganese, Mn": 2.3,               
    "Chromium, Cr": 0.035,              
    "Molybdenum, Mo": 45.0,            
    "Chloride, Cl": 2300.0,                
    "Potassium, K": 4700.0
}

# Функция для перевода в % от суточного потребления

def calculate_pct_dv(nutrient_name, amount):
    if nutrient_name in dv_adults:
        result = (amount / dv_adults[nutrient_name]) * 100
        return int(result)
    return None

In [ ]:
# Оставляем в df_final только нужные по заданию нутриенты

df_final = df_final[df_final['name'].isin(dv_adults.keys())]
df_final.head()

,description,name,amount
4,Hummus,"Phosphorus, P",188.00
5,Hummus,"Manganese, Mn",1.21
6,Hummus,"Potassium, K",326.00
7,Hummus,"Calcium, Ca",46.00
9,Hummus,"Magnesium, Mg",75.90


In [ ]:
# Приводим таблицу к удобному виду

df_final = df_final.groupby(['description', 'name'], as_index=False)['amount'].mean()
df_final = df_final.pivot(index = 'description', columns = 'name', values = 'amount')
df_final = df_final.reset_index()
df_final

name,description,Biotin,"Calcium, Ca","Carbohydrate, by difference",Cholesterol,"Choline, total","Copper, Cu","Fiber, total dietary","Iodine, I","Iron, Fe","Magnesium, Mg","Manganese, Mn","Molybdenum, Mo",Niacin,Pantothenic acid,"Phosphorus, P","Potassium, K",Protein,Riboflavin,"Selenium, Se",Thiamin,"Vitamin A, RAE",Vitamin B-12,Vitamin B-6,"Vitamin C, total ascorbic acid",Vitamin D (D2 + D3),Vitamin E (alpha-tocopherol),Vitamin K (phylloquinone),"Zinc, Zn"
0,Egg whites,NaN,8.600000,NaN,NaN,NaN,0.000000,NaN,NaN,0.185000,10.600000,0.000000,NaN,NaN,NaN,0.000000,129.937500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.019375
1,"Egg whites, dried",NaN,104.333333,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,87.626667,0.000000,NaN,NaN,NaN,106.666667,958.733333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.428667
2,Egg yolk,NaN,119.333333,NaN,NaN,NaN,0.000000,NaN,NaN,4.128000,11.226667,0.068133,NaN,NaN,NaN,443.266667,102.466667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.930667
3,"Egg yolks, dried",NaN,274.444444,NaN,NaN,NaN,0.000000,NaN,NaN,9.301111,25.733333,0.000000,NaN,NaN,NaN,981.111111,230.777778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.190000
4,Whole eggs,NaN,54.928571,NaN,NaN,NaN,0.000000,NaN,NaN,1.770714,11.214286,0.000000,NaN,NaN,NaN,189.428571,116.642857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.203571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5231,"tomatoes, crushed, canned, no salt added",NaN,14.300000,NaN,NaN,NaN,0.117000,2.080000,NaN,2.190000,19.900000,0.153000,NaN,NaN,NaN,33.800000,313.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.900000,NaN,NaN,NaN,0.210000
5232,"tomatoes, crushed, canned, salt added",NaN,19.900000,NaN,NaN,NaN,0.102457,1.924286,NaN,2.278571,18.457143,0.152857,NaN,NaN,NaN,32.785714,350.142857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.785714,NaN,NaN,NaN,0.200143
5233,"tomatoes, whole, canned, solids and liquids, w...",NaN,19.827500,NaN,NaN,NaN,0.059375,0.872500,NaN,0.929125,10.751250,0.075150,NaN,NaN,NaN,17.312500,202.937500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.662500,NaN,NaN,NaN,0.119800
5234,"white rice, long grain, unenriched",0.0,4.460000,NaN,NaN,NaN,0.213750,0.148750,NaN,0.140500,26.525000,0.981125,64.175,1.43375,NaN,108.200000,82.262500,NaN,0.08,6.5625,0.065,NaN,NaN,0.057875,NaN,NaN,NaN,NaN,1.353625


In [ ]:
def clean_usda_description(text):
    if pd.isna(text): return ""
    # Убираем всё, что идет до первого тире, если там список нутриентов (типа B12, B6 - )
    if ' - ' in text:
        parts = text.split(' - ')
        # Если в первой части есть цифры (витамины), берем вторую часть
        if any(char.isdigit() for char in parts[0]):
            return parts[1]
    return text

def super_normalize(text):
    if pd.isna(text): return ""
    # Убираем знаки препинания и лишний мусор
    words = str(text).lower().replace(',', '').replace('-', ' ').replace('(', '').replace(')', '').split()
    
    processed_words = []
    for word in words:
        # Стемминг (отрезаем окончания)
        for suffix in ['s', 'es', 'ed', 'ing']:
            if word.endswith(suffix) and len(word) > 4:
                word = word[:-len(suffix)]
                break
        processed_words.append(word)
    
    return " ".join(processed_words)

# 2. Новая мощная функция поиска через подмножества
# Список слов, которые часто встречаются в USDA, но мешают поиску
STOP_WORDS_FOOD = {'raw', 'canned', 'prepared', 'cooked', 'dried', 'frozen', 'fresh', 'with', 'without', 'added', 'plain', 'enriched'}

def find_match_v4(description, clean_mapping_sets):
    if pd.isna(description): return None
    
    # 1. Убираем технический мусор (B12, B6...)
    description = clean_usda_description(description)
    
    # 2. Нормализуем описание
    desc_norm = super_normalize(description)
    desc_words = set(desc_norm.split())
    
    # 3. Ищем совпадения
    for norm_clean_str, original_name in clean_mapping_sets:
        clean_words = set(norm_clean_str.split())
        
        # Если это сложное название (типа "sweet potato"), 
        # требуем, чтобы ВСЕ слова были в описании
        if len(clean_words) > 1:
            if clean_words.issubset(desc_words):
                return original_name
        # Если это просто одно слово (типа "beef"), 
        # достаточно, чтобы оно просто там было
        else:
            if list(clean_words)[0] in desc_words:
                return original_name
                
    return None

# Пересчитываем маппинг
sorted_clean = sorted(df_raw.columns, key=lambda x: len(str(x).split()), reverse=True)
clean_mapping_sets = [(super_normalize(name), name) for name in sorted_clean]

# ПРИМЕНЯЕМ
df_final['clean_name'] = df_final['description'].apply(lambda x: find_match_v4(x, clean_mapping_sets))

# Запускаем заново
df_final['clean_name'] = df_final['description'].apply(lambda x: find_match_v4(x, clean_mapping_sets))
print(f"Новое кол-во уникальных: {df_final['clean_name'].nunique()}")

# 3. Подготовка данных
# Сортируем от длинных названий к коротким (ВАЖНО для приоритета)
sorted_clean = sorted(df_raw.columns, key=lambda x: len(str(x).split()), reverse=True)
clean_mapping_sets = [(super_normalize(name), name) for name in sorted_clean]

# 4. Запуск процесса
df_final['clean_name'] = df_final['description'].apply(lambda x: find_match_v4(x, clean_mapping_sets))

# 5. Итоговая проверка
found_count = df_final['clean_name'].notna().sum()
unique_ingredients = df_final['clean_name'].nunique()

print(f"--- РЕЗУЛЬТАТ ---")
print(f"Всего строк в базе USDA сопоставлено: {found_count}")
print(f"Уникальных ингредиентов из твоего списка найдено: {unique_ingredients}")

Новое кол-во уникальных: 135
--- РЕЗУЛЬТАТ ---
Всего строк в базе USDA сопоставлено: 2918
Уникальных ингредиентов из твоего списка найдено: 135


In [ ]:
nutri_path = "data/nutrition_dv.csv"
df_grouped = df_final.groupby('clean_name').mean(numeric_only=True).reset_index()
df_grouped.loc[df_grouped['clean_name'] == 'honeydew', 'clean_name'] = 'honey'
df_grouped.loc[df_grouped['clean_name'] == 'grape', 'clean_name'] = 'jam'

df_grouped.to_csv(nutri_path, index=False)

nutri_path, df_final


('data/nutrition_dv.csv',
 name                                        description  Biotin  Calcium, Ca  \
 0                                            Egg whites     NaN     8.600000   
 1                                     Egg whites, dried     NaN   104.333333   
 2                                              Egg yolk     NaN   119.333333   
 3                                      Egg yolks, dried     NaN   274.444444   
 4                                            Whole eggs     NaN    54.928571   
 ...                                                 ...     ...          ...   
 5231           tomatoes, crushed, canned, no salt added     NaN    14.300000   
 5232              tomatoes, crushed, canned, salt added     NaN    19.900000   
 5233  tomatoes, whole, canned, solids and liquids, w...     NaN    19.827500   
 5234                 white rice, long grain, unenriched     0.0     4.460000   
 5235                              yogurt, plain, nonfat     NaN   167.375000   
 


## Similar Recipes: формируем таблицу URL без пропусков

Сохраняем `recipes_urls.csv` (колонки: title, url).
Если в исходном датасете нет URL, делаем офлайн-fallback:
- генерируем slug из `title`
- собираем URL как `https://www.epicurious.com/recipes/food/views/<slug>`

Важно: в итоговом CSV **не должно быть пустых URL**.


In [ ]:
def _slugify(title: str) -> str:
    s = (title or "").strip().lower()
    s = re.sub(r"[^a-z0-9\s-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s or "recipe"

# 1) если в датасете есть url/link — используем их, иначе делаем slug-fallback
cols_low = {str(c).lower(): c for c in df_raw.columns}
url_col = None
for cand in ("url", "link", "recipeurl", "recipe_url", "epi_url"):
    if cand in cols_low:
        url_col = cols_low[cand]
        break

titles = df_raw["title"].astype(str).fillna("").tolist()

if url_col is not None:
    urls = df_raw[url_col].astype(str).fillna("").str.strip().tolist()
else:
    urls = [""] * len(titles)

base = "https://www.epicurious.com/recipes/food/views/"
out_urls: List[str] = []
for t, u in zip(titles, urls):
    u = (u or "").strip()
    if u.startswith("http"):
        out_urls.append(u)
    else:
        out_urls.append(base + _slugify(t))

df_urls = pd.DataFrame({"title": titles, "url": out_urls})
df_urls["url"] = df_urls["url"].astype(str).str.strip()

# гарантия: нет пустых url
df_urls = df_urls[df_urls["url"].str.startswith("http")].drop_duplicates(subset=["title"], keep="first")

urls_path = "data/recipes_urls.csv"
df_urls.to_csv(urls_path, index=False)

urls_path, df_urls.head()


('data/recipes_urls.csv',
                                          title  \
 0              Lentil, Apple, and Turkey Wrap    
 1  Boudin Blanc Terrine with Red Onion Confit    
 2                Potato and Fennel Soup Hodge    
 3             Mahi-Mahi in Tomato Olive Sauce    
 4                    Spinach Noodle Casserole    
 
                                                  url  
 0  https://www.epicurious.com/recipes/food/views/...  
 1  https://www.epicurious.com/recipes/food/views/...  
 2  https://www.epicurious.com/recipes/food/views/...  
 3  https://www.epicurious.com/recipes/food/views/...  
 4  https://www.epicurious.com/recipes/food/views/...  )